# Lab3 CXR: round 4, hybrid architectures and head/loss design (5 folds, same splits)

| Run | What changes | Question |
|---|---|---|
| `fusion_cnx_swin_384` | ConvNeXt-S + Swin-S side by side, pooled features concatenated → MLP, trained end to end | does joint CNN + transformer fusion beat ConvNeXt alone? |
| `maxvit_t_384` | MaxViT-Tiny (convolution + local and grid attention in every block) | does a single hybrid model beat ConvNeXt? |
| `hier_cnx_384` | ConvNeXt-S with two heads: normal / pneumonia / COVID, then bacteria vs virus | does a dedicated bacteria/virus head help? |
| `focal_cnx_384` | ConvNeXt-S with focal loss (γ = 2) | does down-weighting easy images help the hard classes? |

All runs: lung crops + CLAHE, mild augmentation, 384 px, 12 epochs, the same 5 folds as `convnext_s_384`, so every result is directly comparable and can be ensembled.
Estimated time on 2 × T4 (rough guesses): fusion ~2.7 h, MaxViT ~1.8 h, hierarchical ~1.1 h, focal ~1.1 h. Order = priority; a step that fails or runs out of time does not stop the rest.

**Settings:** GPU T4 x2, Internet On, secret `KAGGLE_API_TOKEN`.
**Inputs:** `lab3-processed` and the **Output of last run's notebook version** (*Add Input → Your Work*), which holds `folds.csv` and the ConvNeXt / Swin / DenseNet results. (`lab3-work` alone also works, without the Swin and DenseNet rows.)
**Run with** *Save Version → Save & Run All*, then download `lab3_results_r4.zip` from the version's Output tab.

In [ ]:
# A. Competition data through the API (skipped when already there)
import os, glob
RAW = '/kaggle/working/data'
if not glob.glob(f'{RAW}/**/train_data.csv', recursive=True):
    if not os.environ.get('KAGGLE_API_TOKEN'):
        from kaggle_secrets import UserSecretsClient
        os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
    !pip -q install kaggle
    !kaggle competitions download -c nycu-aimi-2026-lab3-cxr -p /kaggle/working
    !unzip -q -o /kaggle/working/nycu-aimi-2026-lab3-cxr.zip -d {RAW} && rm /kaggle/working/nycu-aimi-2026-lab3-cxr.zip
print(sorted(os.listdir(RAW)))

## B. Code (one cell per script)

In [ ]:
import os; os.makedirs('/kaggle/working/code', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/code/cxr_data.py
"""
Training-time data module for the Lab3 CXR competition. Reads the manifest.csv written by
preprocess_lungcrop.py.

Contents
  * CLAHE as a deterministic *preprocessing* step (applied identically at train AND test time),
    with three modes so it can be ablated:  none | clahe | stack
        stack = 3 input channels [raw, CLAHE(clip 2), CLAHE(clip 4)] -> gives ImageNet-pretrained
                backbones different contrast views instead of 3 identical channels.
  * Train augmentation (mild, anatomy-preserving): affine (rot +-10, shift 6%, scale 0.9-1.1),
    brightness/contrast jitter, optional horizontal flip, optional light RandomErasing.
  * Batch-level Mixup (alpha 0.2) + soft-target cross-entropy. CutMix deliberately left out.
  * CXRDataset reading the manifest.

Preview the augmentations (good figure for the report):
    python cxr_data.py --root processed_data --preview aug_preview.jpg --clahe stack
"""

import argparse
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torchvision.transforms import v2 as T

CLASSES = ["normal", "bacteria", "virus", "COVID-19"]  # column order of the submission CSV
_ALIASES = {"normal": 0, "bacteria": 1, "bacterial": 1, "virus": 2, "viral": 2,
            "covid-19": 3, "covid19": 3, "covid": 3, "covid_19": 3}
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def label_to_idx(name: str) -> int:
    key = str(name).strip().lower()
    if key not in _ALIASES:
        raise ValueError(f"Unknown class folder '{name}'. Add it to _ALIASES in cxr_data.py")
    return _ALIASES[key]


# ----------------------------------------------------------------------------------------
# Intensity preprocessing (deterministic)
# ----------------------------------------------------------------------------------------
def apply_clahe_mode(gray: np.ndarray, mode: str = "clahe", clip: float = 2.0, tile: int = 8) -> np.ndarray:
    """uint8 HxW -> uint8 HxWx3."""
    if mode == "none":
        return np.repeat(gray[..., None], 3, axis=2)
    c1 = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(gray)
    if mode == "clahe":
        return np.repeat(c1[..., None], 3, axis=2)
    if mode == "stack":
        c2 = cv2.createCLAHE(clipLimit=2 * clip, tileGridSize=(tile, tile)).apply(gray)
        return np.stack([gray, c1, c2], axis=2)
    raise ValueError(f"clahe mode must be none|clahe|stack, got {mode}")


# ----------------------------------------------------------------------------------------
# Geometric / photometric augmentation (tensor based, torchvision v2)
# ----------------------------------------------------------------------------------------
def build_transforms(img_size: int = 384, train: bool = True, hflip: bool = False,
                     erasing: float = 0.25, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    ops = [T.ToImage(), T.Resize((img_size, img_size), antialias=True)]
    if train:
        ops += [
            T.RandomApply([T.RandomAffine(degrees=10, translate=(0.06, 0.06), scale=(0.9, 1.1), fill=0)], p=0.8),
            T.RandomApply([T.ColorJitter(brightness=0.15, contrast=0.15)], p=0.5),
        ]
        if hflip:
            ops.append(T.RandomHorizontalFlip(p=0.5))
    ops += [T.ToDtype(torch.float32, scale=True), T.Normalize(mean, std)]
    if train and erasing > 0:
        black = [-m / sd for m, sd in zip(mean, std)]  # erase to black, not to the ImageNet-mean grey
        ops.append(T.RandomErasing(p=erasing, scale=(0.02, 0.08), ratio=(0.5, 2.0), value=black))
    return T.Compose(ops)


# ----------------------------------------------------------------------------------------
# Dataset
# ----------------------------------------------------------------------------------------
class CXRDataset(torch.utils.data.Dataset):
    """df rows come from manifest.csv. Returns (image_tensor, label_idx or -1, out_relpath)."""

    def __init__(self, df: pd.DataFrame, root, transform, clahe: str = "clahe", cache: bool = False):
        self.df = df.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform
        self.clahe = clahe
        self.labels = [label_to_idx(l) if (l and sp != "test") else -1
                       for l, sp in zip(self.df["label"].fillna(""), self.df["split"])]
        self._cache = {} if cache else None

    def __len__(self):
        return len(self.df)

    def _load(self, i):
        if self._cache is not None and i in self._cache:
            return self._cache[i]
        p = self.root / self.df.loc[i, "out_relpath"]
        gray = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            raise FileNotFoundError(p)
        img = apply_clahe_mode(gray, self.clahe)
        if self._cache is not None:
            self._cache[i] = img
        return img

    def __getitem__(self, i):
        x = self.transform(self._load(i))  # HxWx3 uint8 numpy -> CxHxW float tensor
        return x, self.labels[i], self.df.loc[i, "out_relpath"]


def _csv_split(name: str):
    n = name.lower()
    for sp in ("train", "val", "test"):
        if sp in n:
            return sp
    return None


def read_label_csvs(label_dir):
    """Find the competition CSVs (e.g. train_data.csv / val_data.csv) and return
    {(split or None, filename): class_name}. Supports one-hot columns (normal, bacteria, virus, COVID-19)
    or a single label/class column. Test/sample-submission CSVs are ignored."""
    label_dir = Path(label_dir)
    out, used = {}, []
    for f in sorted(label_dir.rglob("*.csv")):
        if any(part.startswith(".") or part == "__MACOSX" for part in f.parts) or f.name == "manifest.csv":
            continue
        sp = _csv_split(f.name)
        if sp == "test" or "sample" in f.name.lower() or "submission" in f.name.lower():
            continue
        df = pd.read_csv(f)
        cols = {c.strip().lower(): c for c in df.columns}
        fcol = cols.get("new_filename") or cols.get("filename") or cols.get("file") or cols.get("image") \
            or df.columns[0]
        onehot = [cols.get(c.lower()) for c in CLASSES]
        if all(onehot):
            idx = df[onehot].to_numpy().argmax(1)
            bad = (df[onehot].to_numpy().sum(1) != 1).sum()
            if bad:
                print(f"[warn] {f.name}: {bad} rows are not exactly one-hot")
            labels = [CLASSES[i] for i in idx]
        else:
            lcol = cols.get("label") or cols.get("class") or cols.get("target")
            if lcol is None:
                print(f"[skip] {f.name}: no one-hot or label column found ({list(df.columns)})")
                continue
            labels = [CLASSES[label_to_idx(v)] for v in df[lcol]]
        for fn, lab in zip(df[fcol].astype(str), labels):
            out[(sp, Path(fn).name)] = lab
        used.append(f"{f.relative_to(label_dir)} ({len(df)} rows, split={sp})")
    print("Label CSVs used:", *used, sep="\n  ")
    return out


def load_manifest(root, splits=("train", "val"), label_dir=None) -> pd.DataFrame:
    """Merged train+val (for K-fold) or test. Drops failed rows loudly.
    label_dir: folder with the competition CSVs; needed when images are not in class sub-folders."""
    df = pd.read_csv(Path(root) / "manifest.csv", keep_default_na=False)
    df = df[df["split"].isin(splits)].copy()
    bad = df[df["method"] == "failed"]
    if len(bad):
        print(f"[warn] dropping {len(bad)} failed images:", bad["orig_relpath"].tolist()[:10])
    df = df[df["method"] != "failed"].reset_index(drop=True)
    if label_dir is not None and any(sp != "test" for sp in splits):
        lab = read_label_csvs(label_dir)
        names = df["orig_relpath"].map(lambda p: Path(p).name)
        df["label"] = [lab.get((sp, n), lab.get((None, n), l)) if sp != "test" else l
                       for sp, n, l in zip(df["split"], names, df["label"])]
        missing = df[(df["split"] != "test") & (df["label"] == "")]
        if len(missing):
            raise ValueError(f"{len(missing)} train/val images have no label, e.g. {missing.orig_relpath.tolist()[:5]}")
    if any(sp != "test" for sp in splits):
        tr = df[df["split"] != "test"]
        print("Class counts:\n" + pd.crosstab(tr["split"], tr["label"], margins=True).to_string())
    return df


# ----------------------------------------------------------------------------------------
# Mixup (batch level) + soft-target loss
# ----------------------------------------------------------------------------------------
def mixup_batch(x, y, num_classes=4, alpha=0.2, p=0.5, label_smoothing=0.0):
    """Returns (x_mixed, soft_targets). With prob 1-p returns the batch unchanged (one-hot targets).
    Switch off for the last ~20% of epochs."""
    t = F.one_hot(y, num_classes).float()
    if label_smoothing > 0:
        t = t * (1 - label_smoothing) + label_smoothing / num_classes
    if alpha <= 0 or np.random.rand() > p:
        return x, t
    lam = float(np.random.beta(alpha, alpha))
    lam = max(lam, 1 - lam)  # keep the original image dominant
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], lam * t + (1 - lam) * t[perm]


def soft_ce(logits, soft_targets, class_weight=None):
    logp = F.log_softmax(logits, dim=1)
    if class_weight is not None:
        logp = logp * class_weight[None, :]
    return -(soft_targets * logp).sum(dim=1).mean()


# ----------------------------------------------------------------------------------------
def _preview(root, out, clahe, img_size, n_imgs=4, n_aug=5, hflip=False):
    df = load_manifest(root, ("test",))
    df = df.sample(min(n_imgs, len(df)), random_state=0)
    tf = build_transforms(img_size, train=True, hflip=hflip)
    mean, std = torch.tensor(IMAGENET_MEAN)[:, None, None], torch.tensor(IMAGENET_STD)[:, None, None]
    rows = []
    for row in df.itertuples():
        g = cv2.imread(str(Path(root) / row.out_relpath), cv2.IMREAD_GRAYSCALE)
        base = cv2.resize(apply_clahe_mode(g, "none"), (img_size, img_size))
        cl = cv2.resize(apply_clahe_mode(g, clahe), (img_size, img_size))
        tiles = [base, cl]
        for _ in range(n_aug):
            x = tf(apply_clahe_mode(g, clahe)) * std + mean
            tiles.append((x.clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8))
        rows.append(np.hstack(tiles))
    grid = np.vstack(rows)
    cv2.imwrite(out, cv2.cvtColor(grid, cv2.COLOR_RGB2BGR))
    print(f"saved {out}: columns = original | CLAHE({clahe}) | {n_aug} random train augmentations")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="processed_data")
    ap.add_argument("--preview", default="aug_preview.jpg")
    ap.add_argument("--clahe", default="clahe", choices=["none", "clahe", "stack"])
    ap.add_argument("--img_size", type=int, default=256)
    ap.add_argument("--hflip", action="store_true")
    a = ap.parse_args()
    _preview(a.root, a.preview, a.clahe, a.img_size, hflip=a.hflip)


# ----------------------------------------------------------------------------------------
# Submission
# ----------------------------------------------------------------------------------------
def write_submission(pred, template_csv, out_csv):
    """pred: dict {original test filename (e.g. '1736.jpeg') -> class index 0..3 or class name}.
    Keeps the template's row order exactly, writes integer one-hot columns, refuses to write if
    any template file is missing a prediction (Kaggle rejects null values)."""
    tpl = pd.read_csv(template_csv)
    names = tpl["new_filename"].astype(str)
    missing = [n for n in names if n not in pred]
    if missing:
        raise ValueError(f"{len(missing)} test files have no prediction, e.g. {missing[:5]}")
    idx = [p if isinstance(p, (int, np.integer)) else label_to_idx(p) for p in (pred[n] for n in names)]
    onehot = np.eye(len(CLASSES), dtype=int)[idx]
    sub = pd.DataFrame(onehot, columns=CLASSES)
    sub.insert(0, "new_filename", names.values)
    sub.to_csv(out_csv, index=False)
    assert sub[CLASSES].sum(1).eq(1).all() and not sub.isna().any().any()
    print(f"wrote {out_csv}: {len(sub)} rows | predicted counts: "
          + ", ".join(f"{c}={int(v)}" for c, v in sub[CLASSES].sum().items()))
    return sub

In [ ]:
%%writefile /kaggle/working/code/train.py
#!/usr/bin/env python
"""
5-fold training for the Lab3 CXR competition (Normal / Bacteria / Virus / COVID-19, macro-F1).

  * merged train+val -> StratifiedKFold (folds.csv is created once and reused by every model)
  * timm backbone (default ConvNeXt-Small IN-22k, 384 px), AdamW + warmup/cosine, AMP, EMA
  * label-smoothed CE, COVID oversampled x5, mild augmentation + CLAHE (cxr_data.py)
  * one fold per run (free Colab friendly): checkpoint every epoch -> resumes after a disconnect
  * --finalize: out-of-fold (OOF) report, per-class logit-offset tuning for macro-F1,
                fold-averaged test predictions -> submission CSVs

Colab (T4):
  !pip -q install timm
  !python train.py --proc_root /content/processed_data --data_root $DATA --work_dir $WORK --fold 0
  ... --fold 1 ... --fold 4      (or --fold all)
  !python train.py --proc_root /content/processed_data --data_root $DATA --work_dir $WORK --finalize \
                   --template_csv $DATA/test_data_sample.csv
"""

import argparse
import copy
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

sys.path.insert(0, str(Path(__file__).resolve().parent))
from cxr_data import (CLASSES, CXRDataset, build_transforms, load_manifest,  # noqa: E402
                      mixup_batch, soft_ce, write_submission)

try:
    from tqdm.auto import tqdm as _tqdm
except ImportError:
    _tqdm = None
NO_BAR = os.environ.get("NO_BAR", "0") == "1"


def tqdm(it, **kw):
    """progress bar, or the plain iterable when NO_BAR=1 (log files) / tqdm missing"""
    return it if (NO_BAR or _tqdm is None) else _tqdm(it, **kw)

SHORT = ["N", "B", "V", "C"]


# --------------------------------------------------------------------------------------
# utils
# --------------------------------------------------------------------------------------
def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def macro_f1(y, pred):
    return f1_score(y, pred, average="macro", labels=list(range(4)), zero_division=0)


def per_class_f1(y, pred):
    return f1_score(y, pred, average=None, labels=list(range(4)), zero_division=0)


def atomic_save(obj, path):
    tmp = str(path) + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


class EMA:
    """Exponential moving average of weights (with warm-up so early epochs are not frozen)."""

    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.updates = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


def get_folds(work_dir, df, n_folds, seed):
    """folds.csv is the single source of truth so every model uses identical splits."""
    path = Path(work_dir) / ("folds.csv" if n_folds == 5 else f"folds{n_folds}.csv")
    if path.exists():
        f = pd.read_csv(path)
        if set(f["out_relpath"]) != set(df["out_relpath"]):
            raise ValueError(f"{path} does not match the current manifest - delete it to re-split")
        return df.merge(f[["out_relpath", "fold"]], on="out_relpath")
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    df = df.copy()
    df["fold"] = -1
    for k, (_, va) in enumerate(skf.split(df, df["label"])):
        df.loc[df.index[va], "fold"] = k
    df[["out_relpath", "orig_relpath", "split", "label", "fold"]].to_csv(path, index=False)
    print(f"Created {path}\n" + pd.crosstab(df["fold"], df["label"]).to_string())
    return df


def make_loader(ds, bs, shuffle, workers, drop_last=False):
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=workers,
                                       pin_memory=torch.cuda.is_available(), drop_last=drop_last,
                                       persistent_workers=workers > 0)


@torch.no_grad()
def predict(model, loader, device, tta=False, desc="predict"):
    model.eval()
    out = []
    for x, _, _ in tqdm(loader, desc=desc, leave=False, dynamic_ncols=True):
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            lg = model(x).float()
            if tta:
                lg = (lg + model(torch.flip(x, dims=[3])).float()) / 2
        out.append(lg.cpu())
    return torch.cat(out).numpy()


# --------------------------------------------------------------------------------------
# models
# --------------------------------------------------------------------------------------
class XRVDenseNet(torch.nn.Module):
    """TorchXRayVision DenseNet121 pretrained on ~8 public chest X-ray datasets, new 4-class head.
    Expects 1-channel input in [-1024, 1024]; the transform's mean/std below produce exactly that
    (pixel/255 -> (x - 0.5) * 2048) and the 3 identical channels are averaged."""

    def __init__(self, weights="densenet121-res224-all", pretrained=True, drop=0.2):
        super().__init__()
        import torchxrayvision as xrv
        base = xrv.models.DenseNet(weights=weights if pretrained else None)
        self.features = base.features
        self.head = torch.nn.Sequential(torch.nn.Dropout(drop), torch.nn.Linear(1024, 4))
        self.pretrained_cfg = {"mean": (0.5, 0.5, 0.5), "std": (1 / 2048,) * 3}

    def forward_features(self, x):
        return F.relu(self.features(x.mean(1, keepdim=True)))

    def forward_head(self, f):
        return self.head(F.adaptive_avg_pool2d(f, 1).flatten(1))

    def forward(self, x):
        return self.forward_head(self.forward_features(x))


CNX = "convnext_small.fb_in22k_ft_in1k_384"
SWIN = "swin_small_patch4_window7_224.ms_in22k_ft_in1k"


def _timm(name, pretrained, drop_path, img_size, **kw):
    import timm
    if name.startswith("swin_"):  # Swin needs the input size up front; the 224 weights are resampled to it
        kw.update(img_size=img_size, window_size=max(7, img_size // 32))
    return timm.create_model(name, pretrained=pretrained, drop_path_rate=drop_path, **kw)


class FusionNet(torch.nn.Module):
    """Late fusion: a CNN and a transformer run side by side on the same image; their pooled
    embeddings (768 + 768) are concatenated and classified by a small MLP. Trained end to end."""

    def __init__(self, a=CNX, b=SWIN, pretrained=True, drop=0.2, drop_path=0.2, img_size=384):
        super().__init__()
        self.a = _timm(a, pretrained, drop_path, img_size, num_classes=0)
        self.b = _timm(b, pretrained, drop_path, img_size, num_classes=0)
        dim = self.a.num_features + self.b.num_features
        self.head = torch.nn.Sequential(torch.nn.LayerNorm(dim), torch.nn.Dropout(drop), torch.nn.Linear(dim, 512),
                                        torch.nn.GELU(), torch.nn.Dropout(drop), torch.nn.Linear(512, 4))
        self.pretrained_cfg = self.a.pretrained_cfg  # both use ImageNet mean/std

    def forward(self, x):
        return self.head(torch.cat([self.a(x), self.b(x)], dim=1))


class HierNet(torch.nn.Module):
    """Hierarchical head on one backbone. Head 1: normal / pneumonia / COVID-19. Head 2: bacteria vs
    virus. Output = 4 log-probabilities: log P(normal), log P(pneu) + log P(bact | pneu),
    log P(pneu) + log P(virus | pneu), log P(COVID). Cross-entropy on this output equals
    CE(head 1) + CE(head 2 on pneumonia images), so training, ensembling and offsets are unchanged."""

    def __init__(self, name=CNX, pretrained=True, drop=0.2, drop_path=0.2, img_size=384):
        super().__init__()
        self.backbone = _timm(name, pretrained, drop_path, img_size, num_classes=0)
        f = self.backbone.num_features
        self.head1 = torch.nn.Sequential(torch.nn.Dropout(drop), torch.nn.Linear(f, 3))
        self.head2 = torch.nn.Sequential(torch.nn.Dropout(drop), torch.nn.Linear(f, 256), torch.nn.GELU(),
                                         torch.nn.Dropout(drop), torch.nn.Linear(256, 2))
        self.pretrained_cfg = self.backbone.pretrained_cfg

    def forward_features(self, x):
        return self.backbone.forward_features(x)

    def forward_head(self, feats):
        z = self.backbone.forward_head(feats, pre_logits=True)
        g, b = F.log_softmax(self.head1(z).float(), 1), F.log_softmax(self.head2(z).float(), 1)
        return torch.stack([g[:, 0], g[:, 1] + b[:, 0], g[:, 1] + b[:, 1], g[:, 2]], dim=1)

    def forward(self, x):
        return self.forward_head(self.forward_features(x))


def build_model(name, pretrained=True, drop=0.2, drop_path=0.2, img_size=384):
    """timm model name, 'xrv_densenet121[:weights]', 'fusion' (ConvNeXt-S + Swin-S) or 'hier[:<timm name>]'."""
    if name == "fusion":
        return FusionNet(pretrained=pretrained, drop=drop, drop_path=drop_path, img_size=img_size)
    if name.startswith("hier"):
        return HierNet(name.split(":", 1)[1] if ":" in name else CNX, pretrained, drop, drop_path, img_size)
    if name.startswith("xrv_densenet121"):
        tag = name.split(":", 1)[1] if ":" in name else "densenet121-res224-all"
        return XRVDenseNet(tag, pretrained, drop)
    import timm
    kw = {}
    if name.startswith("swin_"):  # Swin needs the input size up front; the 224 weights are resampled to it
        kw = {"img_size": img_size, "window_size": max(7, img_size // 32)}
    return timm.create_model(name, pretrained=pretrained, num_classes=4, drop_rate=drop, drop_path_rate=drop_path, **kw)


def safe_load(path, device):
    """None for missing / empty / truncated checkpoints (e.g. a write that failed on a full Drive)."""
    try:
        if not Path(path).exists() or Path(path).stat().st_size < 1024:
            return None
        return torch.load(path, map_location=device, weights_only=False)
    except Exception as e:  # noqa: BLE001
        print(f"[warn] ignoring unreadable checkpoint {path}: {e!r}")
        return None


# --------------------------------------------------------------------------------------
# training one fold
# --------------------------------------------------------------------------------------
def train_fold(args, df, test_df, fold, device):
    fdir = Path(args.work_dir) / args.run_name / f"fold{fold}"
    fdir.mkdir(parents=True, exist_ok=True)
    if (fdir / "DONE.json").exists() and not args.overwrite:
        print(f"[fold {fold}] already done -> skip ({fdir / 'DONE.json'})")
        return
    seed_all(args.seed + fold)

    tr_df, va_df = df[df.fold != fold].reset_index(drop=True), df[df.fold == fold].reset_index(drop=True)
    # COVID oversampling: repeat the rare class inside each epoch
    if args.covid_oversample > 1:
        cov = tr_df[tr_df.label == "COVID-19"]
        tr_df = pd.concat([tr_df] + [cov] * (args.covid_oversample - 1), ignore_index=True)
    if args.hard_neg_csv and args.hard_neg_oversample > 1:
        # e.g. adult non-COVID films that the model confuses with COVID: repeat them so that
        # "adult-looking" stops being a shortcut for COVID (validation images are never repeated)
        hn = set(pd.read_csv(args.hard_neg_csv)["out_relpath"])
        hard = tr_df[tr_df.out_relpath.isin(hn) & (tr_df.label != "COVID-19")].drop_duplicates("out_relpath")
        tr_df = pd.concat([tr_df] + [hard] * (args.hard_neg_oversample - 1), ignore_index=True)
        print(f"[fold {fold}] hard negatives in train: {len(hard)} (x{args.hard_neg_oversample})")

    model = build_model(args.model, args.pretrained, args.drop, args.drop_path, args.img_size)
    cfg = model.pretrained_cfg
    mean, std = cfg.get("mean", (0.485, 0.456, 0.406)), cfg.get("std", (0.229, 0.224, 0.225))
    model = model.to(device).to(memory_format=torch.channels_last)

    tf_tr = build_transforms(args.img_size, True, hflip=args.hflip, erasing=args.erasing, mean=mean, std=std)
    tf_ev = build_transforms(args.img_size, False, mean=mean, std=std)
    ds_tr = CXRDataset(tr_df, args.proc_root, tf_tr, clahe=args.clahe)
    ds_va = CXRDataset(va_df, args.proc_root, tf_ev, clahe=args.clahe)
    ds_te = CXRDataset(test_df, args.proc_root, tf_ev, clahe=args.clahe)
    dl_tr = make_loader(ds_tr, args.batch_size, True, args.num_workers, drop_last=True)
    dl_va = make_loader(ds_va, args.batch_size * 2, False, args.num_workers)

    head = [p for n, p in model.named_parameters() if n.startswith(("head", "fc", "classifier"))]
    head_ids = {id(p) for p in head}
    body = [p for p in model.parameters() if id(p) not in head_ids]
    opt = torch.optim.AdamW([{"params": body, "lr": args.lr},
                             {"params": head, "lr": args.lr * args.head_lr_mult}], weight_decay=args.wd)
    steps_per_epoch = max(1, len(dl_tr) // args.accum)
    total, warm = args.epochs * steps_per_epoch, args.warmup_epochs * steps_per_epoch
    min_ratio = args.min_lr / args.lr

    def lr_lambda(step):
        if step < warm:
            return (step + 1) / max(1, warm)
        t = (step - warm) / max(1, total - warm)
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * min(t, 1.0)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    ema = EMA(model, args.ema_decay) if args.ema_decay > 0 else None

    start_epoch, best, history = 0, {"f1": -1.0, "epoch": -1}, []
    # big, frequently overwritten checkpoints live on LOCAL disk (Drive keeps every overwritten
    # version in its Trash, which counts against the quota). Only small results + one fp16 copy of
    # the best weights go to work_dir.
    cdir = Path(args.ckpt_dir) / args.run_name / f"fold{fold}"
    cdir.mkdir(parents=True, exist_ok=True)
    last_path, best_path = cdir / "last.pt", cdir / "best.pt"
    if not last_path.exists() and (fdir / "last.pt").exists():  # checkpoint left by the old version
        last_path = fdir / "last.pt"
    if not best_path.exists() and (fdir / "best.pt").exists():
        best_path = fdir / "best.pt"
    ck = None if args.overwrite else safe_load(last_path, device)
    if ck is not None and safe_load(best_path, "cpu") is None:
        print(f"[fold {fold}] last checkpoint found but no readable best.pt -> starting fresh")
        ck = None
    if ck is not None:
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        if ema is not None and ck.get("ema") is not None:
            ema.module.load_state_dict(ck["ema"])
            ema.updates = ck["ema_updates"]
        start_epoch, best, history = ck["epoch"] + 1, ck["best"], ck["history"]
        print(f"[fold {fold}] resumed from epoch {start_epoch} (best mF1 {best['f1']:.4f} @ ep {best['epoch']})")

    y_va = np.array(ds_va.labels)
    print(f"[fold {fold}] train {len(ds_tr)} (incl. COVID x{args.covid_oversample}) | val {len(ds_va)} | "
          f"{args.model} @ {args.img_size}px | bs {args.batch_size}x{args.accum} | {args.epochs} ep | {device}")
    mix_off_epoch = int(args.epochs * 0.8)

    for epoch in range(start_epoch, args.epochs):
        t0 = time.time()
        model.train()
        run_loss, n_seen = 0.0, 0
        opt.zero_grad(set_to_none=True)
        pbar = tqdm(dl_tr, desc=f"fold {fold} ep {epoch + 1}/{args.epochs}", leave=False, dynamic_ncols=True)
        for it, (x, y, _) in enumerate(pbar):
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            y = torch.as_tensor(y, device=device)
            alpha = args.mixup_alpha if epoch < mix_off_epoch else 0.0
            x, t = mixup_batch(x, y, 4, alpha=alpha, p=args.mixup_p, label_smoothing=args.label_smoothing)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                out = model(x).float()
                if args.focal_gamma > 0:  # focal loss: down-weight images the model already gets right
                    logp = F.log_softmax(out, 1)
                    loss = -(t * (1 - logp.exp()).pow(args.focal_gamma) * logp).sum(1).mean() / args.accum
                else:
                    loss = soft_ce(out, t) / args.accum
            scaler.scale(loss).backward()
            if (it + 1) % args.accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
                if ema is not None:
                    ema.update(model)
            run_loss += loss.item() * args.accum * x.size(0)
            n_seen += x.size(0)
            if hasattr(pbar, "set_postfix"):
                pbar.set_postfix(loss=f"{run_loss / n_seen:.4f}", lr=f"{opt.param_groups[0]['lr']:.1e}")

        eval_model = ema.module if ema is not None else model
        logits = predict(eval_model, dl_va, device, desc="val")
        pred = logits.argmax(1)
        vloss = F.cross_entropy(torch.from_numpy(logits), torch.from_numpy(y_va)).item()
        f1 = macro_f1(y_va, pred)
        pcf = per_class_f1(y_va, pred)
        rec = {"epoch": epoch + 1, "train_loss": run_loss / max(n_seen, 1), "val_loss": vloss,
               "val_acc": float((pred == y_va).mean()), "val_macro_f1": f1,
               **{f"f1_{c}": float(v) for c, v in zip(SHORT, pcf)}, "lr": opt.param_groups[0]["lr"],
               "sec": time.time() - t0}
        history.append(rec)
        star = ""
        if f1 > best["f1"]:
            best = {"f1": float(f1), "epoch": epoch + 1}
            best_path = cdir / "best.pt"
            atomic_save({"state_dict": eval_model.state_dict(), "args": vars(args), "epoch": epoch + 1}, best_path)
            oof = va_df[["out_relpath", "orig_relpath", "label"]].copy()
            oof["y"] = y_va
            oof[[f"logit_{c}" for c in SHORT]] = logits
            oof["fold"] = fold
            oof.to_csv(fdir / "oof.csv", index=False)
            star = " *best"
        print(f"[fold {fold}] ep {epoch + 1:>2}/{args.epochs} | train {rec['train_loss']:.4f} | val {vloss:.4f} "
              f"acc {rec['val_acc']:.3f} mF1 {f1:.4f} [" + " ".join(f"{c} {v:.2f}" for c, v in zip(SHORT, pcf))
              + f"] | {rec['sec']:.0f}s{star}", flush=True)
        pd.DataFrame(history).to_csv(fdir / "history.csv", index=False)
        last_path = cdir / "last.pt"
        atomic_save({"model": model.state_dict(), "ema": ema.module.state_dict() if ema else None,
                     "ema_updates": ema.updates if ema else 0, "opt": opt.state_dict(),
                     "sched": sched.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch,
                     "best": best, "history": history}, last_path)

    # ---- test predictions with the best checkpoint ----
    eval_model = ema.module if ema is not None else model
    best_sd = torch.load(best_path, map_location=device, weights_only=False)["state_dict"]
    eval_model.load_state_dict(best_sd)
    dl_te = make_loader(ds_te, args.batch_size * 2, False, args.num_workers)
    te_logits = predict(eval_model, dl_te, device, tta=args.tta, desc="test")
    np.save(fdir / "test_logits.npy", te_logits)
    test_df[["out_relpath", "orig_relpath"]].to_csv(fdir / "test_names.csv", index=False)
    if args.tta:  # keep OOF consistent with test-time TTA
        oof = pd.read_csv(fdir / "oof.csv")
        oof[[f"logit_{c}" for c in SHORT]] = predict(eval_model, dl_va, device, tta=True, desc="val-tta")
        oof.to_csv(fdir / "oof.csv", index=False)
    json.dump({"fold": fold, "best_epoch": best["epoch"], "best_val_macro_f1": best["f1"],
               "args": vars(args)}, open(fdir / "DONE.json", "w"), indent=2)
    if args.save_weights:  # one compact copy (~100 MB for ConvNeXt-S) for later Grad-CAM / re-inference
        torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in best_sd.items()},
                   fdir / "best_fp16.pt")
    for f in {cdir / "last.pt", cdir / "best.pt", fdir / "last.pt", fdir / "best.pt"}:
        if f.exists() and not args.keep_last:
            f.unlink()
    print(f"[fold {fold}] done: best mF1 {best['f1']:.4f} @ epoch {best['epoch']} -> {fdir}")


# --------------------------------------------------------------------------------------
# finalize: OOF report, offset tuning, submissions
# --------------------------------------------------------------------------------------
def log_softmax_np(z):
    z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))


def tune_offsets(logp, y, classes=(1, 2), grid=np.arange(-2.0, 2.0001, 0.05), passes=4):
    """Coordinate ascent on per-class additive offsets (normal fixed at 0) to maximise macro-F1."""
    b = np.zeros(4)
    best = macro_f1(y, (logp + b).argmax(1))
    for _ in range(passes):
        changed = False
        for c in classes:
            for v in grid:
                bb = b.copy()
                bb[c] = v
                s = macro_f1(y, (logp + bb).argmax(1))
                if s > best + 1e-9:
                    best, b, changed = s, bb, True
        if not changed:
            break
    return b, best


def report(name, y, pred):
    pcf = per_class_f1(y, pred)
    print(f"\n{name}: macro-F1 {macro_f1(y, pred):.4f} | acc {(pred == y).mean():.4f} | per-class F1 "
          + ", ".join(f"{c} {v:.3f}" for c, v in zip(CLASSES, pcf)))
    cm = pd.DataFrame(confusion_matrix(y, pred, labels=range(4)), index=[f"true {c}" for c in CLASSES],
                      columns=[f"pred {c}" for c in CLASSES])
    print(cm.to_string())


def finalize(args):
    rdir = Path(args.work_dir) / args.run_name
    folds = sorted(int(p.parent.name[4:]) for p in rdir.glob("fold*/DONE.json"))
    if not folds:
        sys.exit(f"No finished folds in {rdir}")
    print(f"Finished folds: {folds}" + ("" if len(folds) == args.n_folds else "  (partial! OOF covers only these)"))
    oof = pd.concat([pd.read_csv(rdir / f"fold{k}" / "oof.csv") for k in folds], ignore_index=True)
    lcols = [f"logit_{c}" for c in SHORT]
    logp, y = log_softmax_np(oof[lcols].to_numpy()), oof["y"].to_numpy()

    report("OOF argmax", y, logp.argmax(1))
    classes = (1, 2, 3) if args.tune_covid else (1, 2)
    b, tuned = tune_offsets(logp, y, classes)
    report(f"OOF tuned offsets {np.round(b, 2).tolist()}", y, (logp + b).argmax(1))

    # honest estimate: tune on the other folds, evaluate on the held-out fold
    if len(folds) > 1:
        raw, nested = [], []
        for k in folds:
            m = oof["fold"].to_numpy() == k
            bk, _ = tune_offsets(logp[~m], y[~m], classes)
            raw.append(macro_f1(y[m], logp[m].argmax(1)))
            nested.append(macro_f1(y[m], (logp[m] + bk).argmax(1)))
        print(f"\nNested check (offsets tuned on other folds): argmax {np.mean(raw):.4f} -> tuned "
              f"{np.mean(nested):.4f} (mean over folds). If tuned <= argmax, submit the argmax file.")

    # test: average fold log-probs
    names = pd.read_csv(rdir / f"fold{folds[0]}" / "test_names.csv")
    te = np.mean([log_softmax_np(np.load(rdir / f"fold{k}" / "test_logits.npy")) for k in folds], axis=0)
    fn = names["orig_relpath"].map(lambda p: Path(p).name).tolist()
    np.save(rdir / "test_logp_mean.npy", te)
    json.dump({"offsets": b.tolist(), "oof_macro_f1_tuned": tuned, "folds": folds},
              open(rdir / "offsets.json", "w"), indent=2)
    if not args.template_csv:
        print("\n(no --template_csv given -> submissions not written)")
        return
    out_a, out_t = rdir / "submission_argmax.csv", rdir / "submission_tuned.csv"
    sub_a = write_submission(dict(zip(fn, te.argmax(1).tolist())), args.template_csv, out_a)
    sub_t = write_submission(dict(zip(fn, (te + b).argmax(1).tolist())), args.template_csv, out_t)
    print("Expected test counts from the spec priors: ~ normal 316, bacteria 556, virus 299, COVID-19 12")
    # sanity check only (NOT used for prediction): COVID files are numbered 0-56 in train/val
    for nm, sub in (("argmax", sub_a), ("tuned", sub_t)):
        num = sub["new_filename"].str.extract(r"^(\d+)")[0].astype(int)
        cov = sub["COVID-19"] == 1
        print(f"[sanity, {nm}] predicted COVID: {int(cov.sum())} | of those numbered <=57: "
              f"{int((cov & (num <= 57)).sum())} | files numbered <=57 not predicted COVID: "
              f"{int((~cov & (num <= 57)).sum())}")


# --------------------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--proc_root", required=True, help="processed_data folder (manifest.csv)")
    ap.add_argument("--data_root", required=True, help="original dataset folder with train_data.csv / val_data.csv")
    ap.add_argument("--work_dir", required=True, help="where folds.csv, checkpoints, OOF go (use Drive)")
    ap.add_argument("--run_name", default="convnext_s_384")
    ap.add_argument("--fold", default="0", help="fold index, comma list, or 'all'")
    ap.add_argument("--n_folds", type=int, default=5)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--finalize", action="store_true")
    ap.add_argument("--template_csv", default="")
    ap.add_argument("--tune_covid", action="store_true", help="also tune the COVID offset (risky with 46 cases)")
    # model / data
    ap.add_argument("--model", default="convnext_small.fb_in22k_ft_in1k_384")
    ap.add_argument("--pretrained", type=int, default=1)
    ap.add_argument("--img_size", type=int, default=384)
    ap.add_argument("--clahe", default="clahe", choices=["none", "clahe", "stack"])
    ap.add_argument("--hflip", action="store_true")
    ap.add_argument("--erasing", type=float, default=0.25)
    ap.add_argument("--covid_oversample", type=int, default=5)
    ap.add_argument("--hard_neg_csv", default="", help="CSV with out_relpath of non-COVID images to oversample")
    ap.add_argument("--hard_neg_oversample", type=int, default=5)
    ap.add_argument("--mixup_alpha", type=float, default=0.0, help="0 = off (baseline); try 0.2")
    ap.add_argument("--mixup_p", type=float, default=0.5)
    ap.add_argument("--label_smoothing", type=float, default=0.1)
    ap.add_argument("--drop", type=float, default=0.2)
    ap.add_argument("--drop_path", type=float, default=0.2)
    # optimisation
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--warmup_epochs", type=int, default=1)
    ap.add_argument("--batch_size", type=int, default=24)
    ap.add_argument("--accum", type=int, default=1)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--head_lr_mult", type=float, default=10.0)
    ap.add_argument("--min_lr", type=float, default=1e-6)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--clip", type=float, default=2.0)
    ap.add_argument("--focal_gamma", type=float, default=0.0, help="0 = cross-entropy; 2 = focal loss")
    ap.add_argument("--ema_decay", type=float, default=0.999, help="0 = off")
    ap.add_argument("--tta", action="store_true", help="hflip TTA (only sensible together with --hflip)")
    ap.add_argument("--num_workers", type=int, default=2)
    ap.add_argument("--overwrite", action="store_true")
    ap.add_argument("--keep_last", action="store_true", help="keep the full checkpoints after the fold finishes")
    ap.add_argument("--prepare", action="store_true", help="only create folds.csv and exit")
    ap.add_argument("--ckpt_dir", default="/content/ckpt" if Path("/content").is_dir() else
                    ("/tmp/ckpt" if Path("/kaggle").is_dir() else ""),
                    help="LOCAL disk for per-epoch checkpoints (default /content/ckpt on Colab)")
    ap.add_argument("--save_weights", type=int, default=1, help="save best weights as fp16 to work_dir")
    args = ap.parse_args()

    Path(args.work_dir, args.run_name).mkdir(parents=True, exist_ok=True)
    if not args.ckpt_dir:
        args.ckpt_dir = str(Path(args.work_dir) / "_ckpt")
    if args.finalize:
        finalize(args)
        return

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("[warn] no GPU found - Colab: Runtime > Change runtime type > T4 GPU | Kaggle: Settings > Accelerator")
    torch.backends.cudnn.benchmark = True
    df = load_manifest(args.proc_root, ("train", "val"), label_dir=args.data_root)
    df = get_folds(args.work_dir, df, args.n_folds, args.seed)
    if args.prepare:
        # also download the pretrained weights ONCE here, so the parallel per-GPU jobs don't race on the same file
        if args.pretrained:
            build_model(args.model, True, args.drop, args.drop_path, args.img_size)
        print("folds + pretrained weights ready")
        return
    test_df = load_manifest(args.proc_root, ("test",))
    folds = range(args.n_folds) if args.fold == "all" else [int(f) for f in args.fold.split(",")]
    for k in folds:
        train_fold(args, df, test_df, k, device)
        if device == "cuda":
            torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/analyze.py
#!/usr/bin/env python
"""
Analysis of one finished run (needs its folds' oof.csv / history.csv / best_fp16.pt).

Writes PNG figures + summary.json to <work_dir>/analysis/<run>/ :
  confusion.png            counts and row-% (argmax and, if present, tuned offsets)
  curves.png               mean validation macro-F1 / loss / per-class F1 per epoch
  bv_probability.png       P(virus) for true bacteria vs true virus  -> how separable the hard pair is
  errors_B_as_V.png        most confident bacteria -> virus mistakes
  errors_V_as_B.png        most confident virus -> bacteria mistakes
  covid_cases.png          every COVID error (missed or false alarm)
  gradcam_covid.png        Grad-CAM of the out-of-fold model on COVID images (where does it look?)
  gradcam_errors.png       Grad-CAM on the most confident bacteria/virus mistakes
  metadata_probe.txt       can image size / file type alone predict the class? (shortcut check)
  by_crop_method.csv       accuracy split by preprocessing method (lungs / mirror / center)

python analyze.py --work_dir WORK --proc_root PROC --run convnext_s_384
"""

import argparse
import json
import sys
from pathlib import Path

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402
from sklearn.metrics import confusion_matrix  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parent))
from cxr_data import CLASSES, apply_clahe_mode, build_transforms  # noqa: E402
from train import SHORT, build_model, log_softmax_np, macro_f1, per_class_f1  # noqa: E402


def load(work, run):
    rdir = Path(work) / run
    folds = sorted(int(p.parent.name[4:]) for p in rdir.glob("fold*/DONE.json"))
    if not folds:
        raise SystemExit(f"no finished folds in {rdir}")
    oof = pd.concat([pd.read_csv(rdir / f"fold{k}" / "oof.csv") for k in folds], ignore_index=True)
    hist = pd.concat([pd.read_csv(rdir / f"fold{k}" / "history.csv").assign(fold=k) for k in folds])
    args = json.load(open(rdir / f"fold{folds[0]}" / "DONE.json"))["args"]
    offsets = np.array(json.load(open(rdir / "offsets.json"))["offsets"]) if (rdir / "offsets.json").exists() else None
    return rdir, folds, oof, hist, args, offsets


def plot_confusion(y, preds, titles, path):
    fig, axes = plt.subplots(1, len(preds), figsize=(5.2 * len(preds), 4.6))
    axes = np.atleast_1d(axes)
    for ax, p, t in zip(axes, preds, titles):
        cm = confusion_matrix(y, p, labels=range(4))
        pct = cm / cm.sum(1, keepdims=True)
        ax.imshow(pct, cmap="Blues", vmin=0, vmax=1)
        for i in range(4):
            for j in range(4):
                ax.text(j, i, f"{cm[i, j]}\n{pct[i, j]:.0%}", ha="center", va="center", fontsize=9,
                        color="white" if pct[i, j] > 0.6 else "black")
        ax.set_xticks(range(4), CLASSES, rotation=20)
        ax.set_yticks(range(4), CLASSES)
        ax.set_xlabel("predicted")
        ax.set_ylabel("true")
        ax.set_title(f"{t}\nmacro-F1 {macro_f1(y, p):.4f}", fontsize=10)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def plot_curves(hist, path):
    m = hist.groupby("epoch").mean(numeric_only=True)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for k, g in hist.groupby("fold"):
        axes[0].plot(g.epoch, g.val_macro_f1, color="0.8", lw=1)
        axes[1].plot(g.epoch, g.val_loss, color="0.8", lw=1)
    axes[0].plot(m.index, m.val_macro_f1, lw=2.5, label="mean of folds")
    axes[1].plot(m.index, m.val_loss, lw=2.5, label="val (mean)")
    axes[1].plot(m.index, m.train_loss, lw=1.5, ls="--", label="train (mean)")
    for c, nm in zip(SHORT, CLASSES):
        axes[2].plot(m.index, m[f"f1_{c}"], lw=2, label=nm)
    for ax, t in zip(axes, ["validation macro-F1", "loss", "validation F1 per class (mean of folds)"]):
        ax.set_title(t)
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def plot_bv(oof, prob, path):
    pv = prob[:, 2] / (prob[:, 1] + prob[:, 2] + 1e-9)  # P(virus | bacteria or virus)
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = np.linspace(0, 1, 41)
    for lab, col in (("bacteria", "tab:orange"), ("virus", "tab:blue")):
        ax.hist(pv[oof.label == lab], bins=bins, alpha=0.55, label=f"true {lab}", color=col)
    ax.axvline(0.5, color="k", ls="--", lw=1)
    ax.set_xlabel("P(virus) / (P(virus) + P(bacteria))  (out-of-fold)")
    ax.set_ylabel("images")
    ax.set_title("How separable bacteria and virus are")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def gallery(rows, proc_root, path, title, ncol=6, extra=None):
    if len(rows) == 0:
        return
    n = len(rows)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.9 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (_, r) in zip(axes, rows.iterrows()):
        img = cv2.imread(str(Path(proc_root) / r.out_relpath), cv2.IMREAD_GRAYSCALE)
        ax.imshow(img, cmap="gray")
        if extra is not None:
            ax.imshow(extra[r.name], cmap="jet", alpha=0.35)
        ax.set_title(f"{Path(r.orig_relpath).name}\ntrue {r.label} | pred {CLASSES[r.pred]} {r.conf:.2f}", fontsize=7.5)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.35 / nrow))
    fig.savefig(path, dpi=110)
    plt.close(fig)


def gradcam(rdir, rows, proc_root, args, device):
    """Grad-CAM of each image's OUT-OF-FOLD model, for its predicted class."""
    cams = {}
    tf = None
    for k, g in rows.groupby("fold"):
        wpath = rdir / f"fold{k}" / "best_fp16.pt"
        if not wpath.exists():
            wpath = rdir / f"fold{k}" / "best.pt"  # older runs: full-precision checkpoint
        if not wpath.exists():
            print(f"[skip gradcam] fold {k}: no saved weights ({len(g)} images)")
            continue
        model = build_model(args["model"], pretrained=False, drop=args.get("drop", 0.2),
                            drop_path=args.get("drop_path", 0.2), img_size=args.get("img_size", 384))
        sd = torch.load(wpath, map_location="cpu", weights_only=False)
        model.load_state_dict(sd["state_dict"] if "state_dict" in sd else sd)
        model = model.float().to(device).eval()
        cfg = model.pretrained_cfg
        tf = build_transforms(args["img_size"], False, mean=cfg.get("mean"), std=cfg.get("std"))
        for idx, r in g.iterrows():
            gray = cv2.imread(str(Path(proc_root) / r.out_relpath), cv2.IMREAD_GRAYSCALE)
            x = tf(apply_clahe_mode(gray, args.get("clahe", "clahe")))[None].to(device)
            feats = model.forward_features(x)
            feats.retain_grad()
            out = model.forward_head(feats)
            model.zero_grad()
            out[0, int(r.pred)].backward()
            w = feats.grad.mean(dim=(2, 3), keepdim=True)
            cam = F.relu((w * feats).sum(1))[0].detach().cpu().numpy()
            cam = cv2.resize(cam, gray.shape[::-1])
            cams[idx] = cam / (cam.max() + 1e-9)
        del model
    return cams


def metadata_probe(oof, manifest, out_txt):
    """Shortcut check: predict the class from image size / file type only (no pixels)."""
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_predict
    m = oof.merge(manifest[["out_relpath", "orig_w", "orig_h"]], on="out_relpath", how="left")
    X = np.c_[m.orig_w, m.orig_h, m.orig_w / m.orig_h,
              m.orig_relpath.str.lower().str.endswith(".png"), m.orig_relpath.str.lower().str.endswith(".jpg")]
    X = np.nan_to_num(X.astype(float))
    p = cross_val_predict(RandomForestClassifier(300, min_samples_leaf=3, random_state=0), X, m.y, cv=5)
    pcf = per_class_f1(m.y, p)
    txt = ("Metadata-only classifier (width, height, aspect, file type; no pixels), 5-fold CV:\n"
           f"  macro-F1 {macro_f1(m.y, p):.4f} | per-class F1 "
           + ", ".join(f"{c} {v:.3f}" for c, v in zip(CLASSES, pcf))
           + "\nA high COVID F1 here means COVID images can be told apart by their source alone (shortcut risk).\n")
    open(out_txt, "w").write(txt)
    print(txt)
    return {f"metadata_f1_{c}": float(v) for c, v in zip(CLASSES, pcf)}


def signal_checks(oof, manifest, out_csv):
    """Per true class: things that are NOT disease but could still differ between classes (shortcuts)."""
    m = oof.merge(manifest, on="out_relpath", how="left", suffixes=("", "_m"))
    for c in ("orig_w", "orig_h", "x0", "x1", "y0", "y1"):
        m[c] = pd.to_numeric(m[c], errors="coerce") if c in m else np.nan
    if "method" not in m:
        m["method"] = "lungs"
    bw, bh = m.x1 - m.x0, m.y1 - m.y0
    side = np.maximum(bw, bh)
    elongated = np.maximum(bw, bh) / np.minimum(bw, bh).clip(lower=1) > 1.5  # resized, not padded
    m["padded"] = (~elongated) & ((side > m.orig_w + 1) | (side > m.orig_h + 1))
    m["non_jpeg"] = ~m.orig_relpath.str.lower().str.endswith(".jpeg")
    m["aspect"] = m.orig_w / m.orig_h
    t = m.groupby("label").agg(n=("y", "size"), median_w=("orig_w", "median"), median_h=("orig_h", "median"),
                               median_aspect=("aspect", "median"), padded_share=("padded", "mean"),
                               non_jpeg_share=("non_jpeg", "mean"),
                               mirror_or_center=("method", lambda v: float((v != "lungs").mean())))
    t = t.round(3)
    t.to_csv(out_csv)
    print("Signal checks per true class (none of these should matter for the diagnosis):\n" + t.to_string())
    return t


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--work_dir", required=True)
    ap.add_argument("--proc_root", required=True)
    ap.add_argument("--run", required=True)
    ap.add_argument("--gallery_n", type=int, default=18)
    ap.add_argument("--gradcam_errors", type=int, default=12)
    ap.add_argument("--no_gradcam", action="store_true")
    ap.add_argument("--covid_like_thr", type=float, default=0.05)
    a = ap.parse_args()

    rdir, folds, oof, hist, args, offsets = load(a.work_dir, a.run)
    out = Path(a.work_dir) / "analysis" / a.run
    out.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    lp = log_softmax_np(oof[[f"logit_{c}" for c in SHORT]].to_numpy())
    prob = np.exp(lp)
    y = oof["y"].to_numpy()
    oof["pred"] = lp.argmax(1)
    oof["conf"] = prob.max(1)
    print(f"{a.run}: folds {folds}, {len(oof)} OOF images")

    preds, titles = [oof.pred.to_numpy()], ["argmax"]
    if offsets is not None:
        preds.append((lp + offsets).argmax(1))
        titles.append(f"tuned offsets {np.round(offsets, 2).tolist()}")
    plot_confusion(y, preds, titles, out / "confusion.png")
    plot_curves(hist, out / "curves.png")
    plot_bv(oof, prob, out / "bv_probability.png")

    err = oof[oof.pred != oof.y]
    b_as_v = err[(err.label == "bacteria") & (err.pred == 2)].sort_values("conf", ascending=False).head(a.gallery_n)
    v_as_b = err[(err.label == "virus") & (err.pred == 1)].sort_values("conf", ascending=False).head(a.gallery_n)
    covid = oof[((oof.y == 3) | (oof.pred == 3)) & (oof.pred != oof.y)]
    gallery(b_as_v, a.proc_root, out / "errors_B_as_V.png", "Most confident: bacteria predicted as virus")
    gallery(v_as_b, a.proc_root, out / "errors_V_as_B.png", "Most confident: virus predicted as bacteria")
    gallery(covid, a.proc_root, out / "covid_cases.png", "COVID errors (missed or false alarm)")

    manifest = pd.read_csv(Path(a.proc_root) / "manifest.csv", keep_default_na=False)
    manifest = manifest.drop_duplicates("out_relpath")
    by = oof.merge(manifest[["out_relpath", "method"]], on="out_relpath", how="left")
    by_m = by.assign(correct=by.pred == by.y).groupby("method").agg(n=("y", "size"), accuracy=("correct", "mean"))
    by_m.to_csv(out / "by_crop_method.csv")
    print("Accuracy by crop method:\n" + by_m.to_string())
    meta = metadata_probe(oof, manifest, out / "metadata_probe.txt")
    signal_checks(oof, manifest, out / "signal_checks.csv")

    # COVID-like negatives: hard-negative candidates (adult / outside-source non-COVID films)
    oof["p_covid"] = prob[:, 3]
    cand = oof[(oof.y != 3) & ((oof.p_covid >= a.covid_like_thr) |
                               ~oof.orig_relpath.str.lower().str.endswith(".jpeg"))].sort_values("p_covid", ascending=False)
    cand[["out_relpath", "orig_relpath", "label", "p_covid", "pred"]].to_csv(out / "covid_like_negatives.csv", index=False)
    gallery(cand.head(36).assign(conf=cand.head(36).p_covid, pred=3), a.proc_root, out / "covid_like_negatives.png",
            f"Non-COVID films the model finds COVID-like (P(COVID) >= {a.covid_like_thr} or not .jpeg): "
            f"{len(cand)} candidates; check visually, keep the adult ones")
    print(f"COVID-like negatives: {len(cand)} -> {out / 'covid_like_negatives.csv'}")

    # label-noise estimate: errors the model is very sure about
    conf_err = oof[(oof.pred != oof.y) & (oof.conf >= 0.9)]
    print("Confident errors (wrong with p >= 0.9) per true class:\n" + conf_err.label.value_counts().to_string())

    if not a.no_gradcam and not args["model"].startswith(("swin_", "fusion")):  # Grad-CAM needs one conv feature map
        cov_rows = oof[oof.y == 3]
        cams = gradcam(rdir, cov_rows, a.proc_root, args, device)
        gallery(cov_rows.loc[list(cams)], a.proc_root, out / "gradcam_covid.png",
                "Grad-CAM, true COVID images (out-of-fold model, predicted class)", extra=cams)
        e_rows = pd.concat([b_as_v.head(a.gradcam_errors // 2), v_as_b.head(a.gradcam_errors // 2)])
        cams_e = gradcam(rdir, e_rows, a.proc_root, args, device)
        gallery(e_rows.loc[list(cams_e)], a.proc_root, out / "gradcam_errors.png",
                "Grad-CAM, most confident bacteria/virus mistakes", extra=cams_e)

    pcf = per_class_f1(y, oof.pred)
    summary = {"run": a.run, "folds": folds, "n": int(len(oof)), "oof_macro_f1": float(macro_f1(y, oof.pred)),
               **{f"f1_{c}": float(v) for c, v in zip(CLASSES, pcf)},
               "errors": int(len(err)), "bacteria_to_virus": int(((oof.y == 1) & (oof.pred == 2)).sum()),
               "virus_to_bacteria": int(((oof.y == 2) & (oof.pred == 1)).sum()),
               "covid_missed": int(((oof.y == 3) & (oof.pred != 3)).sum()),
               "covid_false_alarm": int(((oof.y != 3) & (oof.pred == 3)).sum()),
               "confident_errors_p90": int(len(conf_err)), "covid_like_negatives": int(len(cand)), **meta}
    json.dump(summary, open(out / "summary.json", "w"), indent=2)
    print(json.dumps(summary, indent=2))
    print(f"\nfigures -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/ensemble.py
#!/usr/bin/env python
"""
Combine several finished runs (each = 5 folds trained by train.py on the same folds.csv).

  * per-run OOF macro-F1 + how often runs disagree (diversity)
  * blend = weighted sum of per-run log-probabilities; weights searched on OOF (or equal)
  * per-class offsets tuned on the blended OOF
  * nested check: weights + offsets tuned on 4 folds, scored on the 5th -> honest estimate
  * writes submissions + a one-line-per-run summary CSV (paste into the report)

python ensemble.py --work_dir WORK --runs convnext_s_384 effv2s_384 xrv_dn121_384 \
                   --template_csv TEMPLATE --name ens_v1
"""

import argparse
import itertools
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
from cxr_data import CLASSES, write_submission  # noqa: E402
from train import SHORT, log_softmax_np, macro_f1, per_class_f1, report, tune_offsets  # noqa: E402


def load_run(rdir, n_folds):
    folds = sorted(int(p.parent.name[4:]) for p in rdir.glob("fold*/DONE.json"))
    if len(folds) != n_folds:
        raise SystemExit(f"{rdir.name}: finished folds {folds}, need all {n_folds} (wrong --n_folds?) -> finish it first")
    oof = pd.concat([pd.read_csv(rdir / f"fold{k}" / "oof.csv") for k in folds], ignore_index=True)
    oof = oof.sort_values("out_relpath").reset_index(drop=True)
    names = pd.read_csv(rdir / f"fold{folds[0]}" / "test_names.csv")
    te = np.mean([log_softmax_np(np.load(rdir / f"fold{k}" / "test_logits.npy")) for k in folds], axis=0)
    args = json.load(open(rdir / f"fold{folds[0]}" / "DONE.json"))["args"]
    return oof, names, te, args


def search_weights(lps, y, grid=(0, 0.5, 1, 1.5, 2), passes=3):
    """coordinate ascent on run weights (first run fixed at 1) for OOF macro-F1 (argmax)"""
    w = np.ones(len(lps))
    blend = lambda ww: sum(wi * lp for wi, lp in zip(ww, lps)) / max(ww.sum(), 1e-9)  # noqa: E731
    best = macro_f1(y, blend(w).argmax(1))
    for _ in range(passes):
        changed = False
        for i in range(1, len(lps)):
            for g in grid:
                ww = w.copy()
                ww[i] = g
                s = macro_f1(y, blend(ww).argmax(1))
                if s > best + 1e-9:
                    best, w, changed = s, ww, True
        if not changed:
            break
    return w / w.sum(), best


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--work_dir", required=True)
    ap.add_argument("--runs", nargs="+", required=True)
    ap.add_argument("--name", default="ensemble")
    ap.add_argument("--template_csv", default="")
    ap.add_argument("--n_folds", type=int, default=5)
    ap.add_argument("--weights", default="search", choices=["search", "equal"])
    ap.add_argument("--tune_covid", action="store_true")
    args = ap.parse_args()

    work = Path(args.work_dir)
    out = work / "ensembles" / args.name
    out.mkdir(parents=True, exist_ok=True)
    runs = {r: load_run(work / r, args.n_folds) for r in args.runs}

    base = runs[args.runs[0]][0]
    for r, (oof, names, _, _) in runs.items():
        if not oof["out_relpath"].equals(base["out_relpath"]):
            raise SystemExit(f"{r}: OOF images differ from {args.runs[0]} - were they trained on the same folds.csv?")
        if not names["out_relpath"].equals(runs[args.runs[0]][1]["out_relpath"]):
            raise SystemExit(f"{r}: test image order differs")
    y, fold = base["y"].to_numpy(), base["fold"].to_numpy()
    lcols = [f"logit_{c}" for c in SHORT]
    lps = [log_softmax_np(runs[r][0][lcols].to_numpy()) for r in args.runs]
    tes = [runs[r][2] for r in args.runs]

    # ---- per-run summary + diversity ----
    rows = []
    for r, lp in zip(args.runs, lps):
        p = lp.argmax(1)
        pcf = per_class_f1(y, p)
        a = runs[r][3]
        rows.append({"run": r, "model": a.get("model"), "img_size": a.get("img_size"), "epochs": a.get("epochs"),
                     "oof_macro_f1": round(macro_f1(y, p), 4), "oof_acc": round(float((p == y).mean()), 4),
                     **{f"f1_{c}": round(float(v), 4) for c, v in zip(CLASSES, pcf)}})
    summ = pd.DataFrame(rows)
    print("\nPer-run OOF (argmax):\n" + summ.to_string(index=False))
    if len(lps) > 1:
        print("\nDisagreement between runs (share of OOF images where argmax differs):")
        for (i, a), (j, b) in itertools.combinations(enumerate(args.runs), 2):
            d = float((lps[i].argmax(1) != lps[j].argmax(1)).mean())
            print(f"  {a} vs {b}: {d:.3f}")

    # ---- blend ----
    eq = np.ones(len(lps)) / len(lps)
    w = search_weights(lps, y)[0] if (args.weights == "search" and len(lps) > 1) else eq
    blend = lambda ww, L: sum(wi * lp for wi, lp in zip(ww, L))  # noqa: E731
    oof_eq, oof_w = blend(eq, lps), blend(w, lps)
    classes = (1, 2, 3) if args.tune_covid else (1, 2)
    b, _ = tune_offsets(oof_w, y, classes)
    print(f"\nWeights: " + ", ".join(f"{r}={wi:.2f}" for r, wi in zip(args.runs, w)))
    report("Ensemble, equal weights, argmax", y, oof_eq.argmax(1))
    report("Ensemble, searched weights, argmax", y, oof_w.argmax(1))
    report(f"Ensemble, searched weights + offsets {np.round(b, 2).tolist()}", y, (oof_w + b).argmax(1))

    # ---- nested check: everything tuned without the scored fold ----
    nest_eq, nest_full = [], []
    for k in np.unique(fold):
        m = fold == k
        ww = search_weights([lp[~m] for lp in lps], y[~m])[0] if (args.weights == "search" and len(lps) > 1) else eq
        bb, _ = tune_offsets(blend(ww, [lp[~m] for lp in lps]), y[~m], classes)
        nest_eq.append(macro_f1(y[m], blend(eq, [lp[m] for lp in lps]).argmax(1)))
        nest_full.append(macro_f1(y[m], (blend(ww, [lp[m] for lp in lps]) + bb).argmax(1)))
    ne, nf = float(np.mean(nest_eq)), float(np.mean(nest_full))
    best_rule = "weighted_tuned" if nf > ne else "equal_argmax"
    print(f"\nNested check (mean over folds): equal+argmax {ne:.4f} | weights+offsets {nf:.4f} -> submit '{best_rule}'")

    summ = pd.concat([summ, pd.DataFrame([
        {"run": f"ENSEMBLE {args.name} (equal, argmax)", "oof_macro_f1": round(macro_f1(y, oof_eq.argmax(1)), 4),
         "nested_macro_f1": round(ne, 4)},
        {"run": f"ENSEMBLE {args.name} (weights+offsets)", "oof_macro_f1": round(macro_f1(y, (oof_w + b).argmax(1)), 4),
         "nested_macro_f1": round(nf, 4)}])], ignore_index=True)
    summ.to_csv(out / "summary.csv", index=False)
    json.dump({"runs": args.runs, "weights": w.tolist(), "offsets": b.tolist(), "nested_equal": ne,
               "nested_weighted_tuned": nf, "recommended": best_rule}, open(out / "ensemble.json", "w"), indent=2)
    print(f"\nsummary -> {out / 'summary.csv'}")

    if args.template_csv:
        fn = runs[args.runs[0]][1]["orig_relpath"].map(lambda p: Path(p).name).tolist()
        te_eq, te_w = blend(eq, tes), blend(w, tes)
        write_submission(dict(zip(fn, te_eq.argmax(1).tolist())), args.template_csv, out / "submission_equal_argmax.csv")
        write_submission(dict(zip(fn, (te_w + b).argmax(1).tolist())), args.template_csv,
                         out / "submission_weighted_tuned.csv")
        print(f"Recommended file: {out / ('submission_' + best_rule + '.csv')}")


if __name__ == "__main__":
    main()

The TA's submission template (file order for the submission):

In [ ]:
# The TA's submission template (test_data_sample.csv) is not part of the Kaggle download.
# Add it as an input (a small private dataset) and copy it here:
import glob, shutil
shutil.copy(glob.glob('/kaggle/input/**/test_data_sample.csv', recursive=True)[0], '/kaggle/working/test_data_sample.csv')

In [ ]:
# 0. Setup
!pip -q install timm torchxrayvision
import glob, os, shutil, subprocess, sys, time, json, traceback
from pathlib import Path
import torch, pandas as pd
from IPython.display import Image, display

def find_all(pattern, root='/kaggle/input'):
    return sorted(glob.glob(f'{root}/**/{pattern}', recursive=True))

CODE = '/kaggle/working/code'
DATA = os.path.dirname(find_all('train_data.csv', '/kaggle/working/data')[0])          # label CSVs
IMG_ROOT = os.path.dirname(find_all('train_images', '/kaggle/working/data')[0])       # original images
TEMPLATE = '/kaggle/working/test_data_sample.csv'
PROC = [os.path.dirname(p) for p in find_all('manifest.csv') if 'masked' not in p][0]  # lab3-processed (lung crops)
WORK = '/kaggle/working/work'
NFOLD = 5
os.makedirs(WORK, exist_ok=True)
for d in sorted({os.path.dirname(p) for p in find_all('folds.csv') if '/code/' not in p},
                key=lambda d: len(glob.glob(f'{d}/*/fold*/DONE.json'))):               # earlier results (lab3-work)
    shutil.copytree(d, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns('last.pt', '*.tmp'))
sys.path.insert(0, CODE)
os.environ['NO_BAR'] = '1'
NGPU = max(torch.cuda.device_count(), 1)
print('DATA', DATA, '\nIMG_ROOT', IMG_ROOT, '\nPROC', PROC, '\nGPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
for run in sorted({Path(p).parent.parent.name for p in glob.glob(f'{WORK}/*/fold*/history.csv')}):
    print(f'  {run}: finished folds', sorted(int(Path(p).parent.name[4:]) for p in glob.glob(f'{WORK}/{run}/fold*/DONE.json')))
assert os.path.exists(f'{WORK}/folds.csv'), 'attach the lab3-work dataset (folds.csv + convnext_s_384)'

In [ ]:
# Helpers
def sh(cmd, tail=6000):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-tail:], r.stderr[-1500:])
    return r.returncode

def run_cv(run_name, extra=''):
    base = (f'python {CODE}/train.py --proc_root {PROC} --data_root {DATA} --work_dir {WORK} --n_folds {NFOLD} '
            f'--run_name {run_name} --num_workers {max(1, 4 // NGPU)} {extra}')
    todo = [f for f in range(NFOLD) if not os.path.exists(f'{WORK}/{run_name}/fold{f}/DONE.json')]
    if todo and sh(base + ' --prepare', 1500) != 0:      # pretrained weights downloaded once, before the parallel jobs
        raise RuntimeError('prepare failed')
    jobs = []
    for g in range(NGPU):
        fl = todo[g::NGPU]
        if fl:
            logf = f'{WORK}/{run_name}_gpu{g}.log'
            print(f'[{run_name}] GPU{g} -> folds {fl}')
            jobs.append((subprocess.Popen(f'CUDA_VISIBLE_DEVICES={g} {base} --fold {",".join(map(str, fl))}', shell=True,
                                          stdout=open(logf, 'a'), stderr=subprocess.STDOUT), logf, [0]))
    def drain():
        for p, logf, pos in jobs:
            with open(logf) as f:
                f.seek(pos[0]); new = f.read(); pos[0] = f.tell()
            for line in new.splitlines():
                if '] ep' in line or 'done' in line or 'Error' in line:
                    print(line)
    t0 = time.time()
    while jobs and any(p.poll() is None for p, _, _ in jobs):
        time.sleep(60)
        drain()
    drain()
    for p, logf, _ in jobs:
        if p.returncode != 0:
            print(f'!! job failed ({logf}):'); print(open(logf).read()[-3000:])
    print(f'[{run_name}] {(time.time() - t0) / 60:.0f} min')
    sh(f'python {CODE}/train.py --proc_root {PROC} --data_root {DATA} --work_dir {WORK} --n_folds {NFOLD} '
       f'--run_name {run_name} --finalize --template_csv {TEMPLATE}', 3500)

def analyze(run_name, show=('confusion', 'curves', 'bv_probability')):
    sh(f'python {CODE}/analyze.py --work_dir {WORK} --proc_root {PROC} --run {run_name}', 2500)
    for f in show:
        p = f'{WORK}/analysis/{run_name}/{f}.png'
        if os.path.exists(p):
            print(f); display(Image(p, width=1000))

def step(fn):                      # a failed step must not stop the commit: later steps and the zip still run
    try:
        fn()
    except Exception:
        traceback.print_exc()

## 1. Train (priority order; finished runs are skipped)

In [ ]:
MODELS = {
    'fusion_cnx_swin_384': '--model fusion --batch_size 8 --accum 3 --lr 5e-5 --epochs 12',
    'maxvit_t_384':        '--model maxvit_tiny_tf_384.in1k --batch_size 12 --accum 2 --lr 1e-4 --epochs 12',
    'hier_cnx_384':        '--model hier --epochs 12',
    'focal_cnx_384':       '--focal_gamma 2 --epochs 12',
}
for name, extra in MODELS.items():
    step(lambda: run_cv(name, extra))
    step(lambda: analyze(name, show=('confusion', 'curves')))

## 2. Compare every run on the same out-of-fold images
`both wrong %` = share of ConvNeXt's errors that this model also gets wrong. Near 100% means it cannot fix ConvNeXt's mistakes in an ensemble.

In [ ]:
from ensemble import load_run
from train import log_softmax_np, macro_f1, per_class_f1, SHORT
from cxr_data import CLASSES
L = [f'logit_{c}' for c in SHORT]
done = [r for r in sorted({Path(p).parent.parent.name for p in glob.glob(f'{WORK}/*/fold*/DONE.json')})
        if len(glob.glob(f'{WORK}/{r}/fold*/DONE.json')) == NFOLD]
lp = {r: log_softmax_np(load_run(Path(WORK) / r, NFOLD)[0][L].to_numpy()) for r in done}
y = load_run(Path(WORK) / 'convnext_s_384', NFOLD)[0].y.to_numpy()
ref_err = lp['convnext_s_384'].argmax(1) != y
rows = []
for r in done:
    p = lp[r].argmax(1); f = per_class_f1(y, p); err = p != y
    a = json.load(open(glob.glob(f'{WORK}/{r}/fold0/DONE.json')[0]))['args']
    rows.append({'run': r, 'model': a['model'], 'epochs': a['epochs'], 'oof_macro_f1': round(macro_f1(y, p), 4),
                 **{c: round(float(v), 3) for c, v in zip(CLASSES, f)},
                 'covid_false_alarm': int(((p == 3) & (y != 3)).sum()), 'covid_missed': int(((p != 3) & (y == 3)).sum()),
                 'errors': int(err.sum()), 'both_wrong_%': round(100 * (err & ref_err).sum() / ref_err.sum(), 1)})
table = pd.DataFrame(rows).sort_values('oof_macro_f1', ascending=False)
table.to_csv(f'{WORK}/round4_table.csv', index=False)
table

## 3. Ensemble: ConvNeXt + every new run (weights and offsets searched on OOF, checked by the nested test)

In [ ]:
NEW = [r for r in MODELS if r in done]
RUNS = ['convnext_s_384'] + NEW
step(lambda: sh(f'python {CODE}/ensemble.py --work_dir {WORK} --n_folds {NFOLD} --runs {" ".join(RUNS)} --name round4 '
                f'--template_csv {TEMPLATE}', 7000))

## 4. Submission files
Written by this run (in `work/`):
- `<run>/submission_tuned.csv` and `<run>/submission_argmax.csv` for every model
- `ensembles/round4/submission_weighted_tuned.csv` and `submission_equal_argmax.csv`

Suggested order: the best single run by OOF (tuned), then the ensemble's recommended file. The leaderboard has only 12 COVID films, so differences under about 1 point are noise.

In [ ]:
best = table.iloc[0].run
shutil.copy(f'{WORK}/{best}/submission_tuned.csv', '/kaggle/working/submission.csv')
print('submission.csv =', best, 'tuned')
os.chdir('/kaggle/working')
!zip -qr lab3_results_r4.zip work submission.csv -x "*.pt" "*.log" "*.tmp"
!ls -la lab3_results_r4.zip
from IPython.display import FileLink
FileLink('lab3_results_r4.zip')